# 11.10 視覺 token 預算怎麼影響能力？


把圖片分得更細，能讓小物件佔多個位置，但會讓整串上下文變長。16×16圖的patch邊長從4變2，不是只多一倍位置，而是上下、左右各多一倍，總共四倍。本節把這種成本先算出來，才知道更細圖片特徵需要多少序列空間。

前置是[10.2切塊數量](https://birdhackor.github.io/tiny-perceptron-vlm/10.2.html)與[3.1多位置注意力](https://birdhackor.github.io/tiny-perceptron-vlm/3.1.html)。視覺token在這裡指一個patch產生、插入模型的一條特徵，不是一個中文字。若文字有T個位置、圖有N個位置，完整自注意力要比較每個位置與其他位置，矩陣大約有(T+N)²格。這個格數是工作量線索，不等於整個模型的精確執行時間。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
height = 16
text_tokens = 20
for patch_size in [8, 4, 2]:
    visual_tokens = (height // patch_size) ** 2
    total = text_tokens + visual_tokens
    print("邊長", patch_size, "視覺位置", visual_tokens, "總長度", total, "注意力格數", total**2)

輸入圖高寬都16、文字位置20。`//`是整數除法，這裡的邊長都整除16；每列塊數平方得到N。輸出三行分別：邊長8有4視覺位置、總長24、576格；邊長4有16位置、總長36、1296格；邊長2有64位置、總長84、7056格。邊長減半讓N四倍，注意力成本還受固定20個文字位置影響。

更細切塊可能幫助讀小字，卻不保證答案更準，因為把像素整理成特徵的工具仍需要訓練，才能保留問題所需的資訊。還需確認總長度沒超過模型的最大上下文，圖佔的格子會擠掉文字空間。如果為符合長度而截斷答案，可能連訓練目標都消失，這個問題與[7.10](https://birdhackor.github.io/tiny-perceptron-vlm/7.10.html)相連。

每塊RGB像素有三個通道；邊長4時一塊有4×4×3=48個數，邊長2時則只有2×2×3=12個數。把這些數混合成固定寬度特徵的加權表，叫像素投影矩陣；依照本課程的PyTorch表格約定，每一行負責一個輸出特徵，每一欄對應一個輸入數，因此有48個輸入欄的舊表不能直接接到只有12個數的新入口。[10.3](https://birdhackor.github.io/tiny-perceptron-vlm/10.3.html)有這個轉換的實例。比較不同切法應準備相應配置與權重，報告訓練預算，不能只改一個推理參數就說細粒度方案失敗。

注意力有因果遮罩時，後面不能偷看未來，一部分格子會被遮住；但常見實現仍建立或處理大尺寸矩陣，平方量級仍是有用預算線索。還有每頭、每層的倍數與投影成本，本表沒有列，因此576格不是整個訓練只做576次運算。它適合比較相同模型下序列長度變化，而不是預測不同硬體的精確秒數。例如，若你設定模型最多接收64個位置，視覺64加文字20得到總長84，就已經超過上限；若上限是128則仍放得下。先查自己配置的具體上限，再比總長，避免最後才發現答案被截掉。

練習只把height改32，保持patch_size列表與文字20不變。先算N分別16、64、256，最後一行總長276、格數76176，再執行核對。圖的邊長翻倍同樣讓面積四倍；這個預算表可幫助你先決定哪些配置適合CPU短實驗。
